In [ ]:
%matplotlib widget
%matplotlib widget

'''
Generates a .pkl file of the model, to be read and imported on matlab
'''
import os
from pathlib import Path
import time
import torch
import numpy as np
import math
import gc
from functools import partial
from dataset import Dataset, load_dataframes_from_folder, reverse_normalization
from torch.utils.data import DataLoader
from transformer_zerostep import GPTConfig, GPT, warmup_cosine_lr, GPT_chopped
import argparse
import warnings
import matplotlib.pyplot as plt
import copy
from collections import OrderedDict
import pickle as pkl


# set figure parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams['axes.labelsize']=14
plt.rcParams['xtick.labelsize']=11
plt.rcParams['ytick.labelsize']=11
plt.rcParams['axes.grid']=True
plt.rcParams['axes.xmargin']=0

In [2]:
# Overall settings
out_dir = "out"

model_name = "ICC_paper.pt"


# Compute settings
cuda_device = "cuda:0"
no_cuda = True
threads = 10
compile = False

# Configure compute
torch.set_num_threads(threads) 
use_cuda = not no_cuda and torch.cuda.is_available()
device_name  = cuda_device if use_cuda else "cpu"
device = torch.device(device_name)
device_type = 'cuda' if 'cuda' in device_name else 'cpu' # for later use in torch.autocast
torch.set_float32_matmul_precision("high")
print(torch.cuda.is_available())
# Create out dir
out_dir = Path(out_dir)
exp_data = torch.load(out_dir/model_name, map_location=device, weights_only=False)
seq_len = exp_data["cfg"].seq_len
nx = exp_data["cfg"].nx
exp_data["iter_num"]
print(seq_len)
print(exp_data["iter_num"])
print(exp_data['best_val_loss'])
print(exp_data["cfg"])
print(exp_data["cfg"].lr)
print(exp_data["train_time"]/3600)
print(exp_data["model_args"])

True
10
21829
0.0023218223359435797
Namespace(model_dir='out', out_file='new_dataset_short_noise_h10', in_file='new_dataset_short_noise_h10', init_from='scratch', seed=42, log_wandb=False, nx=4, nu=6, ny=1, seq_len=10, mag_range=(0.5, 0.97), phase_range=(0.0, 1.5707963267948966), fixed_system=False, n_layer=2, n_head=2, n_embd=32, dropout=0, bias=False, batch_size=128, max_iters=40000, warmup_iters=5000, lr=1e-05, weight_decay=0.0, eval_interval=10, eval_iters=10, fixed_lr=False, threads=16, no_cuda=False, cuda_device='cuda:0', compile=False, beta1=0.9, beta2=0.95, block_size=10, lr_decay_iters=40000, min_lr=1.0000000000000002e-06, decay_lr=True, eval_batch_size=128)
1e-05
4.572892640166812
{'n_layer': 2, 'n_head': 2, 'n_embd': 32, 'n_x': 4, 'n_y': 1, 'n_u': 6, 'block_size': 10, 'bias': False, 'dropout': 0}


In [3]:
# generate the model
model_args = exp_data["model_args"]
gptconf = GPTConfig(**model_args)
model = GPT(gptconf).to(device)


state_dict = exp_data["model"]

keys_raw = state_dict.keys()
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    if k.startswith('module.'):
        state_dict[k[7:]] = v
        state_dict.pop(k)

keys = state_dict.keys()


model.load_state_dict(state_dict)

number of parameters: 0.03M


<All keys matched successfully>

In [4]:
print(len(keys_raw))
print(len(keys))

18
18


In [29]:
# model_name[:-3]

In [8]:
test_input = torch.rand(5,gptconf.block_size,gptconf.n_u)
print(test_input.size())
test_output = model(test_input)
print(test_output.size())
# print(test_output)
gptconf_ord_dict = OrderedDict(gptconf.__dict__)
state_dict.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input, 'test_output': test_output}
state_dict.update(in_out_dict)

state_dict.keys()


data_to_save = state_dict
name_to_save = "./pkl_models/" + model_name[:-3] + '_weights.pkl'
# name_to_save = model_name[:-3] + '_weights.pkl'

with open(name_to_save,'wb') as f:
    pkl.dump(data_to_save, f)



torch.Size([5, 10, 6])
torch.Size([5, 10, 1])
